# MoP-Forge Goal 48 Code Cached-Sparse L4 Report

This notebook runs the next code-dataset efficiency comparison on Google Colab L4 and writes lightweight report artifacts under `reports/goal48_code_cached_sparse_efficiency/`.

It compares Dense, MoP Full, warm sparse, and cached sparse teacher-distilled training on the same fixed code split. Set `QUALITY_FORMAT` to `fixed_code_xml` for the next narrow code-repair quality run. It excludes checkpoints from the report folder.

## 0. Settings

Use **Runtime > Change runtime type > GPU > L4** before running. For a short smoke run, lower `MAX_STEPS`; for a real report, keep the split fixed and use the same eval cadence for every model.

In [ ]:
REPO_URL = "https://github.com/NikanBHMNJ/MoP.git"
REPO_DIR = "/content/MoP"
REPORT_ID = "goal48_code_cached_sparse_efficiency"
REPORT_DIR = f"reports/{REPORT_ID}"

MAX_STEPS = 1000
EVAL_EVERY_STEPS = 100
SAVE_EVERY_STEPS = 500
MAX_TRAIN_EXAMPLES = 10000
MAX_EVAL_EXAMPLES = 512
COUNT_PER_CATEGORY = 100
SPLIT_SEED = 42
QUALITY_FORMAT = "fixed_code_xml"

# Set after reviewing the Dense or MoP Full baseline eval loss.
TARGET_EVAL_LOSS = None

CACHE_TEACHER_TOP_K = 16
CACHE_RECORDS_PER_SHARD = 512
CACHED_DISTILLATION_WEIGHT = 0.2
CACHED_DISTILLATION_TEMPERATURE = 2.0
HARD_EXAMPLE_REPLAY = False
HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD = None
HARD_EXAMPLE_REPLAY_MULTIPLIER = 2

RUN_DENSE = True
RUN_MOP_FULL = True
RUN_WARM_ADAPTER_NORM_HEAD = True
RUN_CACHED_TAIL = True
RUN_CACHED_LORA = False

REQUIRE_CUDA = True


In [ ]:
from __future__ import annotations

import datetime as dt
import json
import os
import re
import shutil
import subprocess
from pathlib import Path


def sh(command: str, *, check: bool = True) -> str:
    print(f"\n$ {command}")
    result = subprocess.run(
        command,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {command}")
    return result.stdout


def read_json(path: str | Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def write_json(path: str | Path, data: dict) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")
    return path


def parse_key(output: str, key: str) -> str:
    match = re.search(rf"^{re.escape(key)}=(.+)$", output, flags=re.MULTILINE)
    if not match:
        raise ValueError(f"missing {key}=... in command output")
    return match.group(1).strip()


## 1. Clone, Install, And Check Runtime

In [ ]:
repo_dir = Path(REPO_DIR)
if not (repo_dir / ".git").exists():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    sh(f"git clone --depth 1 {REPO_URL} {repo_dir}")
os.chdir(repo_dir)

sh("python -m pip install -q -e .[dev,gpu]")
sh("mopforge version")
sh("mopforge doctor")
sh("mopforge runtime detect")
sh("nvidia-smi", check=False)

import torch

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this report notebook.")


## 2. Build Fixed Code Dataset

In [ ]:
dataset_output = sh(
    " ".join(
        [
            "mopforge gpu prepare-efficiency-data",
            f"--count-per-category {COUNT_PER_CATEGORY}",
            f"--split-seed {SPLIT_SEED}",
            "--overwrite",
            f"--quality-format {QUALITY_FORMAT}",
        ]
    )
)
DATASET_REF = parse_key(dataset_output, "dataset_ref")
DATASET_SPLIT_ID = parse_key(dataset_output, "split_id")
print("DATASET_REF=", DATASET_REF)
print("DATASET_SPLIT_ID=", DATASET_SPLIT_ID)


## 3. Generate L4 Configs

In [ ]:
CONFIG_DIR = Path("configs/jobs/colab_l4_goal48_code")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

COMMON_OVERRIDES = {
    "device": "cuda",
    "precision": "bf16",
    "require_device_available": True,
    "dataset_ref": DATASET_REF,
    "dataset_split_id": DATASET_SPLIT_ID,
    "lesson_path": "data/coding_bugfix_efficiency_lessons.jsonl",
    "max_steps": int(MAX_STEPS),
    "eval_every_steps": int(EVAL_EVERY_STEPS),
    "save_every_steps": int(SAVE_EVERY_STEPS),
    "max_train_examples": int(MAX_TRAIN_EXAMPLES),
    "max_eval_examples": int(MAX_EVAL_EXAMPLES),
    "run_generation_eval": True,
    "generation_eval_examples": 8,
    "output_root": "gpu_runs",
    "artifact_root": "artifacts",
}
if TARGET_EVAL_LOSS is not None:
    COMMON_OVERRIDES["target_eval_loss"] = float(TARGET_EVAL_LOSS)


def make_config(label: str, template: str, overrides: dict | None = None) -> Path:
    envelope = read_json(template)
    payload = dict(envelope["payload"])
    payload.update(COMMON_OVERRIDES)
    payload.update(overrides or {})
    payload["name"] = f"{REPORT_ID}_{label}"
    metadata = dict(payload.get("metadata") or {})
    metadata.update(
        {
            "report_id": REPORT_ID,
            "label": label,
            "dataset_ref": DATASET_REF,
            "dataset_split_id": DATASET_SPLIT_ID,
            "split_seed": SPLIT_SEED,
            "target_eval_loss": TARGET_EVAL_LOSS,
            "cache_teacher_top_k": CACHE_TEACHER_TOP_K,
            "cache_records_per_shard": CACHE_RECORDS_PER_SHARD,
            "cached_distillation_weight": CACHED_DISTILLATION_WEIGHT,
            "cached_distillation_temperature": CACHED_DISTILLATION_TEMPERATURE,
            "hard_example_replay": HARD_EXAMPLE_REPLAY,
            "hardware_target": "google_colab_l4",
        }
    )
    payload["metadata"] = metadata
    output = CONFIG_DIR / f"{label}.json"
    write_json(output, {"kind": "gpu_train", "version": envelope.get("version", "1"), "metadata": envelope.get("metadata", {}), "payload": payload})
    return output


dense_config = make_config("dense", "configs/jobs/100m_dense_extended_efficiency.json")
mop_full_config = make_config("mop_full", "configs/jobs/100m_mop_full_extended_efficiency.json")
warm_config = make_config("warm_adapter_norm_head_64", "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json")


## 4. Train Baselines And Warm Sparse

In [ ]:
RUNS: dict[str, dict] = {}


def train_config(label: str, config_path: Path) -> dict:
    sh(f"mopforge gpu validate {config_path}")
    output = sh(f"mopforge gpu train {config_path}")
    run_id = parse_key(output, "run_id")
    checkpoint_path = parse_key(output, "latest_checkpoint_path")
    RUNS[label] = {"run_id": run_id, "checkpoint_path": checkpoint_path, "config_path": str(config_path)}
    return RUNS[label]


if RUN_DENSE:
    train_config("dense", dense_config)
if RUN_MOP_FULL:
    train_config("mop_full", mop_full_config)

if RUN_WARM_ADAPTER_NORM_HEAD:
    if "mop_full" not in RUNS:
        raise RuntimeError("Warm sparse run needs RUN_MOP_FULL=True.")
    warm_config = make_config(
        "warm_adapter_norm_head_64",
        "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json",
        {
            "resume_from_checkpoint": RUNS["mop_full"]["checkpoint_path"],
            "base_checkpoint_path": RUNS["mop_full"]["checkpoint_path"],
            "resume_model_only": True,
            "save_trainable_only_checkpoints": True,
        },
    )
    train_config("warm_adapter_norm_head_64", warm_config)


## 5. Cached Sparse Tail With Teacher Top-k KL

In [ ]:
if RUN_CACHED_TAIL:
    if "mop_full" not in RUNS:
        raise RuntimeError("Cached tail run needs RUN_MOP_FULL=True.")
    cache_path = Path("outputs") / f"{REPORT_ID}_teacher_topk_cache_manifest.json"
    sh(
        " ".join(
            [
                "mopforge gpu cache-activations",
                str(warm_config),
                f"--checkpoint {RUNS['mop_full']['checkpoint_path']}",
                f"--output {cache_path}",
                "--dtype bf16",
                f"--teacher-top-k {CACHE_TEACHER_TOP_K}",
                f"--records-per-shard {CACHE_RECORDS_PER_SHARD}",
            ]
        )
    )
    cached_config = make_config(
        "cached_warm_adapter_norm_head_64",
        "configs/jobs/100m_mop_warm_adapters_norm_head_64_colab_efficiency.json",
        {
            "resume_from_checkpoint": RUNS["mop_full"]["checkpoint_path"],
            "base_checkpoint_path": RUNS["mop_full"]["checkpoint_path"],
            "resume_model_only": True,
            "save_trainable_only_checkpoints": True,
            "activation_cache_path": str(cache_path),
            "offload_frozen_backbone_for_cache": True,
            "distillation_enabled": True,
            "distillation_weight": float(CACHED_DISTILLATION_WEIGHT),
            "distillation_temperature": float(CACHED_DISTILLATION_TEMPERATURE),
            "distillation_top_k": int(CACHE_TEACHER_TOP_K),
            "hard_example_replay_enabled": bool(HARD_EXAMPLE_REPLAY),
            "hard_example_replay_loss_threshold": HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD,
            "hard_example_replay_multiplier": int(HARD_EXAMPLE_REPLAY_MULTIPLIER),
        },
    )
    train_config("cached_warm_adapter_norm_head_64", cached_config)


## 6. Write Lightweight Report Folder

In [ ]:
REPORT_PATH = Path(REPORT_DIR)
REPORT_PATH.mkdir(parents=True, exist_ok=True)

LIGHTWEIGHT_FILES = ["config.json", "gpu_training_result.json", "metrics.json", "runtime.json", "state.json", "memory_estimate.json"]


def copy_lightweight_run(label: str, run_id: str) -> None:
    src = Path("gpu_runs") / run_id
    dest = REPORT_PATH / "runs" / label
    dest.mkdir(parents=True, exist_ok=True)
    for name in LIGHTWEIGHT_FILES:
        path = src / name
        if path.exists():
            shutil.copy2(path, dest / name)


for label, record in RUNS.items():
    copy_lightweight_run(label, record["run_id"])

write_json(REPORT_PATH / "run_manifest.json", RUNS)
write_json(
    REPORT_PATH / "experiment_settings.json",
    {
        "report_id": REPORT_ID,
        "created_utc": dt.datetime.utcnow().replace(microsecond=0).isoformat() + "Z",
        "dataset_ref": DATASET_REF,
        "dataset_split_id": DATASET_SPLIT_ID,
        "split_seed": SPLIT_SEED,
        "quality_format": QUALITY_FORMAT,
        "max_steps": MAX_STEPS,
        "eval_every_steps": EVAL_EVERY_STEPS,
        "target_eval_loss": TARGET_EVAL_LOSS,
        "cache_teacher_top_k": CACHE_TEACHER_TOP_K,
        "cache_records_per_shard": CACHE_RECORDS_PER_SHARD,
        "cached_distillation_weight": CACHED_DISTILLATION_WEIGHT,
        "cached_distillation_temperature": CACHED_DISTILLATION_TEMPERATURE,
        "hard_example_replay": HARD_EXAMPLE_REPLAY,
        "hard_example_replay_loss_threshold": HARD_EXAMPLE_REPLAY_LOSS_THRESHOLD,
        "hard_example_replay_multiplier": HARD_EXAMPLE_REPLAY_MULTIPLIER,
        "hardware_target": "google_colab_l4",
    },
)

run_ids = [record["run_id"] for record in RUNS.values()]
if len(run_ids) >= 2:
    sh(
        " ".join(
            [
                "mopforge gpu compare-runs",
                *run_ids,
                f"--output {REPORT_PATH / 'comparison.json'}",
                f"--output-csv {REPORT_PATH / 'comparison.csv'}",
            ]
        )
    )


In [ ]:
rows = []
comparison_path = REPORT_PATH / "comparison.json"
if comparison_path.exists():
    rows = read_json(comparison_path).get("runs", [])


def cell(value):
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.4f}"
    return str(value)


table = [
    "| Run | Train loss | Eval loss | Best eval loss | Time-to-target sec | Tokens/sec | Peak reserved VRAM | Trainable ratio | Checkpoint MB | Distill top-k | Offloaded params | Exact match | Verifier pass | Syntax pass | Device |",
    "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |",
]
for row in rows:
    table.append(
        "| "
        + " | ".join(
            [
                cell(row.get("run_id")),
                cell(row.get("train_loss")),
                cell(row.get("eval_loss")),
                cell(row.get("best_eval_loss")),
                cell(row.get("time_to_target_loss_sec")),
                cell(row.get("tokens_per_sec")),
                cell(row.get("peak_reserved_gb")),
                cell(row.get("trainable_param_ratio")),
                cell(row.get("checkpoint_size_mb")),
                cell(row.get("distillation_top_k")),
                cell(row.get("cached_backbone_offloaded_param_count")),
                cell(row.get("gen_exact_match_rate")),
                cell(row.get("gen_verifier_pass_rate")),
                cell(row.get("gen_syntax_pass_rate")),
                cell(row.get("selected_device")),
            ]
        )
        + " |"
    )

readme = f"""# Goal 48 Code Cached-Sparse L4 Efficiency Report

This report was generated by `notebooks/colab_l4_goal48_code_cached_sparse_report.ipynb`.

## Purpose

Compare Dense, MoP Full, warm sparse, and cached sparse teacher-distilled training on a fixed code split.

## Scope

- Dataset ref: `{DATASET_REF}`
- Split ID: `{DATASET_SPLIT_ID}`
- Split seed: `{SPLIT_SEED}`
- Quality format: `{QUALITY_FORMAT}`
- Target hardware: Google Colab L4
- Target eval loss: `{TARGET_EVAL_LOSS}`
- Cache teacher top-k: `{CACHE_TEACHER_TOP_K}`
- Hard-example replay: `{HARD_EXAMPLE_REPLAY}`

## Results

{chr(10).join(table)}

## Artifacts

- `comparison.json`
- `comparison.csv`
- `run_manifest.json`
- `experiment_settings.json`
- `runs/<label>/` lightweight run metadata

Checkpoint and model weight files are intentionally excluded from this report folder.

## Interpretation Rules

This report supports only measured claims. A cached sparse run is a same-quality efficiency win only if it remains close to Dense eval loss and generated-code quality while improving a named efficiency axis such as VRAM, trainable parameters, checkpoint size, or time-to-target-loss.
"""

(REPORT_PATH / "README.md").write_text(readme, encoding="utf-8")
print((REPORT_PATH / "README.md").read_text(encoding="utf-8"))
